# Evaluacion de modelos IA para cobranzas

Esta fase evalua dos componentes del sistema: prediccion de mora por factura/corte y clustering de clientes. La entrada oficial viene de `03_preparacion/outputs/`, usando el split por `factura_id` definido en preparacion.

## Flujo de la fase

1. Validar entradas oficiales de preparacion.
2. Ejecutar benchmarking supervisado: baseline, regresion logistica, Random Forest y XGBoost si esta disponible.
3. Revisar overfitting/underfitting con gaps train-test y curva de aprendizaje.
4. Evaluar fairness por grupos de negocio sensibles o proxies.
5. Ejecutar clustering usando la base por cliente preparada en fase 3 y exportar perfiles interpretables.

In [ ]:
from __future__ import annotations

import subprocess
import sys
from pathlib import Path

import pandas as pd

cwd = Path.cwd().resolve()
PROJECT_DIR = cwd if (cwd / "AGENTS.md").exists() else next(p for p in cwd.parents if (p / "AGENTS.md").exists())
PHASE_DIR = PROJECT_DIR / "04_evaluacion_modelos_ia"
PREP_OUTPUTS = PROJECT_DIR / "03_preparacion" / "outputs"
OUTPUTS = PHASE_DIR / "outputs"
OUTPUTS.mkdir(exist_ok=True)

MODEL_SCRIPT = PHASE_DIR / "evaluacion_modelos_cobranzas.py"
CLUSTER_SCRIPT = PHASE_DIR / "clustering_clientes_cobranzas.py"

required_inputs = [
    PREP_OUTPUTS / "features_ml_prepared.csv",
    PREP_OUTPUTS / "train_facturas_ids.csv",
    PREP_OUTPUTS / "test_facturas_ids.csv",
    PREP_OUTPUTS / "features_selected.csv",
    PREP_OUTPUTS / "client_features_clustering_base.csv",
    PREP_OUTPUTS / "client_clustering_features_selected.csv",
]
missing = [p for p in required_inputs if not p.exists()]
if missing:
    raise FileNotFoundError("Faltan entradas oficiales: " + ", ".join(str(p) for p in missing))

print(PROJECT_DIR)
print(f"Entradas validadas: {len(required_inputs)}")

## Entrada preparada

La preparacion conserva trazabilidad (`factura_id`, `cliente_id`, `fecha_corte`), target (`target_mora`) y predictores autorizados en `features_selected.csv`.

In [ ]:
df = pd.read_csv(PREP_OUTPUTS / "features_ml_prepared.csv")
features = pd.read_csv(PREP_OUTPUTS / "features_selected.csv")
train_ids = pd.read_csv(PREP_OUTPUTS / "train_facturas_ids.csv")
test_ids = pd.read_csv(PREP_OUTPUTS / "test_facturas_ids.csv")

pd.DataFrame({
    "metrica": ["filas", "columnas", "facturas", "clientes", "features", "facturas_train", "facturas_test"],
    "valor": [len(df), df.shape[1], df["factura_id"].nunique(), df["cliente_id"].nunique(), len(features), len(train_ids), len(test_ids)],
})

## Componente 1: benchmarking supervisado

El script compara modelos, calcula metricas por train/test, exporta matriz de confusion, fairness, curva de aprendizaje y un artefacto `joblib` del mejor modelo.

In [ ]:
def run_script(script_path: Path) -> None:
    result = subprocess.run(
        [sys.executable, str(script_path)],
        cwd=PROJECT_DIR,
        capture_output=True,
        text=True,
        encoding="utf-8",
        errors="replace",
    )
    print(result.stdout[-5000:])
    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError(f"Fallo {script_path.name}")

run_script(MODEL_SCRIPT)

In [ ]:
benchmark = pd.read_csv(OUTPUTS / "benchmark_metrics.csv")
gap = pd.read_csv(OUTPUTS / "train_test_gap.csv")
display(benchmark)
display(gap[["model", "f1_macro_train", "f1_macro_test", "gap_f1_macro", "diagnosis"]])

## Fairness y curva de aprendizaje

Fairness se revisa por `sector_dominante`, `tiene_garantia` y `tiene_disputa_activa` cuando existen grupos con tamano suficiente. La curva usa una validacion interna separada por `factura_id`.

In [ ]:
fairness = pd.read_csv(OUTPUTS / "fairness_gap_summary.csv")
learning_curve = pd.read_csv(OUTPUTS / "learning_curve_best_model.csv")
display(fairness)
display(learning_curve)

## Componente 2: clustering de clientes

El clustering consume `client_features_clustering_base.csv`, compara KMeans y DBSCAN, y exporta asignaciones y perfiles de cluster en escala de negocio interpretable.

In [ ]:
run_script(CLUSTER_SCRIPT)

In [ ]:
cluster_metrics = pd.read_csv(OUTPUTS / "clustering_metrics.csv")
cluster_profiles = pd.read_csv(OUTPUTS / "cluster_profiles.csv")
display(cluster_metrics.head(12))
display(cluster_profiles)

## Outputs esperados

Todos los artefactos tecnicos de esta fase deben quedar dentro de `04_evaluacion_modelos_ia/outputs/`.

In [ ]:
expected_outputs = [
    "benchmark_metrics.csv",
    "train_test_gap.csv",
    "classification_report_best_model.txt",
    "confusion_matrix_best_model.csv",
    "class_metrics_best_model.csv",
    "fairness_by_group.csv",
    "fairness_gap_summary.csv",
    "learning_curve_best_model.csv",
    "learning_curve_best_model.png",
    "best_model_artifact.joblib",
    "model_feature_schema.csv",
    "model_metadata.json",
    "clustering_metrics.csv",
    "client_cluster_assignments.csv",
    "cluster_profiles.csv",
    "clustering_model_features.csv",
    "summary.txt",
]

pd.DataFrame({
    "output": expected_outputs,
    "existe": [(OUTPUTS / name).exists() for name in expected_outputs],
})